# 新增两基线：十配置运行与评估

使用既有训练、测试划分，只运行 PeakVI、DiTSim。各方法通过完整区域轴、逐 Subclass 配额和输出范围核验后立即评分并发布。发布同时刷新 CSV、结果 README 和全量比较图。服务器若已有后台队列通过独立入口运行，应使用「新增基线评分与发布」处理已完成生成任务。

In [ ]:
from pathlib import Path
import os,sys,json,subprocess
import numpy as np
import pandas as pd
import anndata as ad
from threadpoolctl import threadpool_limits

PROJECT=Path('/home/sccasimp/program/xb/simscDNAm').resolve()
OUTPUTS=Path('/home/sccasimp/data/xb/outputs/simscDNAm').resolve()
os.chdir(PROJECT)
sys.path.insert(0,str(PROJECT))
from data.paths import require_environment
from data.baseline import prepare_imputed_training
require_environment()
NEW=['PeakVI','DiTSim']
OLD=['simscDNAm','FittedMarginal','simATAC','simCAS']
SUBSETS={'liu2023':['GSE213221','GSE215213','GSE215342'],
         'liu2021':['GSE130553','GSE131354','GSE131360'],
         'tian2023':['GSE167119','GSE167577','GSE167978']}
CASES=['liu2023/full']+[f'{study}/{accession}' for study,accessions in SUBSETS.items() for accession in accessions]
COLUMNS=['A1 1-WD↓','A1 MSE↓','A1 Pearson↑','A2 Pearson↑','A3 miLISI↑']
DEVICE=os.environ.get('NEW_BASELINE_DEVICE','cuda:1')
ENV={**os.environ,'OMP_NUM_THREADS':'8','MKL_NUM_THREADS':'8','OPENBLAS_NUM_THREADS':'8','NUMBA_NUM_THREADS':'8','PYTHONUNBUFFERED':'1'}
RUN=OUTPUTS/'internal/new_baselines'
RUN.mkdir(parents=True,exist_ok=True)
for case in CASES:
    prepared=OUTPUTS/'preprocessing'/case
    prepare_imputed_training(prepared/'TrainingReal.h5ad',prepared/'HeldoutReal.h5ad',prepared/'imputed_baselines')
print('十配置训练填补缓存已准备；正式模型参数不变。',flush=True)

import nbformat
publication={}
for cell in nbformat.read(PROJECT/'notebooks/新增基线评分与发布.ipynb',as_version=4).cells[:-1]:
    if cell.cell_type=='code':exec(cell.source,publication)


In [ ]:
def generated_path(case,method):
    return OUTPUTS/'baseline'/case/method/'data.h5ad'

def verify_generated(case,method):
    prepared=OUTPUTS/'preprocessing'/case
    real=ad.read_h5ad(prepared/'HeldoutReal.h5ad',backed='r')
    generated=ad.read_h5ad(generated_path(case,method),backed='r')
    try:
        assert generated.shape==real.shape and generated.var_names.equals(real.var_names)
        pd.testing.assert_series_equal(generated.obs['Subclass'].astype(str).value_counts().sort_index(),
            real.obs['Subclass'].astype(str).value_counts().sort_index(),check_names=False)
        assert generated.obs_names.is_unique
        for start in range(0,generated.n_obs,32):
            block=np.asarray(generated.X[start:start+32])
            assert np.isfinite(block).all() and block.min()>=0 and block.max()<=1
    finally:
        generated.file.close();real.file.close()

for case in CASES:
    for method in NEW:
        record=RUN/f"{case.replace('/','__')}__{method}.json"
        if generated_path(case,method).is_file() and record.exists() and json.loads(record.read_text()).get('状态')=='完成':
            verify_generated(case,method)
            publication['score'](case,method)
            publication['publish']()
            continue
        log=RUN/f"{case.replace('/','__')}__{method}.log"
        command=[sys.executable,'-m',f'baseline.{method}.inference','--dataset',case]
        if method in ['PeakVI','DiTSim']:command+=['--device',DEVICE]
        print('正式训练',case,method,flush=True)
        with log.open('w') as stream:
            result=subprocess.run(command,cwd=PROJECT,env=ENV,stdout=stream,stderr=subprocess.STDOUT)
        record.write_text(json.dumps({'状态':'完成' if result.returncode==0 else '失败','退出码':result.returncode,'日志':str(log)},ensure_ascii=False,indent=2))
        if result.returncode!=0:raise RuntimeError(f'{case} {method} 官方接口失败，查看 {log}')
        verify_generated(case,method)
        publication['score'](case,method)
        publication['publish']()
print('三基线十配置训练、生成与配额范围核验完成。',flush=True)
